# Qwen3.8-27B Abliterated Q4 — Kaggle dual-T4 proof of life

**Model:** [`huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF`](https://huggingface.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF), exact file `Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf` (~16.6 GB decimal). This is Huihui's **UD-DW** variant, not the regular Qwen weights. The author says only a subset of layers were abliterated. Do not assume its coding/tool-call quality is identical to the base model.

**Objective for this notebook only:** verify both T4s, build a current CUDA-enabled llama.cpp, download *only* the selected GGUF, boot a **localhost-only** OpenAI-compatible API with **32,768 requested context**, generate a real coding reply, and probe tool calling. No MCP, tunnel, SSH, public server, Git clone of your own project, or remote file access is configured.

**Kaggle settings:** Accelerator **GPU T4 ×2**, **Internet ON**, Python. Run the cells in order. Model and engine live under `/kaggle/tmp/` (ephemeral scratch), to keep your ~20 GiB `/kaggle/working` saved-output allowance free for small reports. This means they **must be downloaded/built again after the runtime is wiped**. Do not use *Save & Run All* for this interactive long-running server experiment.

If a step fails, **stop there and share its error**. This notebook has not been executed on your Kaggle account or GPUs; passing the static checks only means the file is structurally valid, not that the model works on your hardware.

## 1 — Hardware / CUDA / disk preflight

Must see **two NVIDIA T4s**, the CUDA compiler, and enough scratch space. `nvidia-smi` reports allocated VRAM later; two cards are *not* a single 30 GiB memory pool.

In [7]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
SCRATCH = Path('/kaggle/tmp/qwen38-abliterated')
SCRATCH.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_home')

smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader'],
    text=True, capture_output=True, check=True,
).stdout.strip()
print('GPUs (index, name, total MiB, free MiB):\n' + smi)
gpu_lines = [line for line in smi.splitlines() if line.strip()]
if len(gpu_lines) != 2 or not all('T4' in line.upper() for line in gpu_lines):
    raise RuntimeError('Stop: select the GPU T4 x2 accelerator in Kaggle session options.')

cuda_bin = Path('/usr/local/cuda/bin')
if cuda_bin.is_dir():
    os.environ['PATH'] = str(cuda_bin) + os.pathsep + os.environ.get('PATH', '')
for exe in ('git', 'cmake', 'nvcc'):
    if not shutil.which(exe):
        raise RuntimeError(f'Stop: required executable {exe!r} is missing. Share the preflight output.')
subprocess.run(['nvcc', '--version'], check=True)
subprocess.run(['cmake', '--version'], check=True)

free_gib = shutil.disk_usage(SCRATCH).free / 1024**3
print(f'Free scratch disk: {free_gib:.1f} GiB')
if free_gib < 24:
    raise RuntimeError('Stop: less than 24 GiB scratch disk available; model/build may not fit.')
print('PREFLIGHT PASSED — no model has been downloaded yet.')

GPUs (index, name, total MiB, free MiB):
0, Tesla T4, 15360 MiB, 14912 MiB
1, Tesla T4, 15360 MiB, 14912 MiB
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).
Free scratch disk: 1118.7 GiB
PREFLIGHT PASSED — no model has been downloaded yet.


## 2 — Build a current llama.cpp for T4 CUDA (SM 75)

We build only `llama-server` with two compile jobs to limit memory spikes. The source commit hash is printed for reproducibility. Compilation can take several minutes. No PyTorch/Transformers model weights are loaded.

In [8]:

import subprocess
from pathlib import Path

# Build llama.cpp for Kaggle's dual T4 GPUs
SCRATCH = Path("/kaggle/tmp/qwen38-abliterated")
SOURCE = SCRATCH / "llama.cpp"
BUILD = SOURCE / "build"

SCRATCH.mkdir(parents=True, exist_ok=True)

# Clone only if the repository is not already present
if not SOURCE.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp.git",
        str(SOURCE)
    ], check=True)

# Record the exact version
commit = subprocess.run(
    ["git", "-C", str(SOURCE), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("llama.cpp commit:", commit)

# Configure CUDA build
# GGML_CUDA_NO_VMM avoids the missing CUDA::cuda_driver target
subprocess.run([
    "cmake",
    "-S", str(SOURCE),
    "-B", str(BUILD),
    "-DGGML_CUDA=ON",
    "-DGGML_CUDA_NO_VMM=ON",
    "-DCMAKE_CUDA_ARCHITECTURES=75",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DLLAMA_BUILD_TESTS=OFF",
], check=True)

# Compile llama-server
print("Building llama-server...")

subprocess.run([
    "cmake",
    "--build", str(BUILD),
    "--target", "llama-server",
    "--parallel", "2"
], check=True)

# Verify the binary
SERVER = BUILD / "bin" / "llama-server"

if not SERVER.is_file():
    raise RuntimeError(
        "Build finished without llama-server. Do not continue."
    )

subprocess.run([str(SERVER), "--version"], check=True)
subprocess.run([str(SERVER), "--list-devices"], check=True)

# Check required inference options
help_text = subprocess.run(
    [str(SERVER), "--help"],
    capture_output=True,
    text=True,
    check=True
).stdout

required_flags = [
    "--split-mode",
    "--tensor-split",
    "--cache-type-k",
    "--ctx-size",
    "--jinja"
]

missing = [
    flag for flag in required_flags
    if flag not in help_text
]

if missing:
    raise RuntimeError(
        f"Missing required llama.cpp options: {missing}"
    )

print("ENGINE BUILD PASSED.")
print("llama-server:", SERVER)


llama.cpp commit: 2b129ccfa03aea330d2d9ac4650a10de393dbe3a
-- llama.cpp version: 0.5.0-dev


CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.25.3
-- ggml commit:  2b129cc
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.3s)
-- Generating done (0.3s)
-- Build files have been written to: /kaggle/tmp/qwen38-abliterated/llama.cpp/build
Building llama-server...
[  2%] Built target vendor-hash
[  5%] Built target ggml-base
[  5%] Built target cpp-httplib
[  5%] Built target llama-common-base
[  6%] Provisioning UI assets
-- UI: HF stamp matches 'ggm

0.00.000.915 I srv  llama_server: initializing ...
version: 0.5.0-dev (build 1, commit 2b129cc)
built with GNU 11.4.0 for Linux x86_64
0.00.000.781 I srv  llama_server: initializing ...


Available devices:
  CUDA0: Tesla T4 (14911 MiB, 14806 MiB free)
  CUDA1: Tesla T4 (14911 MiB, 14806 MiB free)
ENGINE BUILD PASSED.
llama-server: /kaggle/tmp/qwen38-abliterated/llama.cpp/build/bin/llama-server


In [9]:
!ps -eo pid,etime,stat,%cpu,%mem,args --sort=-%cpu | head -20
!free -h

    PID     ELAPSED STAT %CPU %MEM COMMAND
   5279       03:06 Sl    2.9  0.8 /usr/bin/python3 -m pylsp
     61       56:21 Ssl   0.5  0.0 /opt/bin/nvidia-smi -l 1 --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv,noheader
     12       56:41 Sl    0.3  0.5 /usr/bin/python3 /usr/local/bin/jupyter-server --ip=* --NotebookApp.base_url=/k/353184215/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidHlwIjoiSldUIn0..Q--27tJefb4j--dYw-DSsA.SgtFDjm-jGvh_SeomXfPTYRyZ8wErZS1EO-CQYto88NgKT0aJ5J1Xw5jqx8DHejWd2-e1zy49xPYRxslug028feJzSoLWVYBHhYkhICiPFsbhpRjroz-DC5OO7q8GOVR9U0q5QBi1jIljoCko5mqznlAbkitVaLN4eHaGDje2kS8nSHdZUYpRd7ujXIPJcnWmto2tHRadgNTI8tXo2OsfyzICNiaUdOM8LHcfG7tKZYyICGlpiUZs6_XzILkxXVd.pKxRiN_Zin2y4YYowomDUw/proxy/ --NotebookApp.token= --NotebookApp.notebook_dir=/kaggle/working --NotebookApp.allow_origin=* --NotebookApp.disable_check_xsrf=True --NotebookApp.iopub_data_rate_limit=10000000000 --NotebookApp.open_browser=False --NotebookApp.port=8888 --allow-root --Mult

## 3 — Download the exact Huihui abliterated Q4 model

The checkpoint is approximately **16.6 GB**. We intentionally download neither the 54+ GB BF16 weights nor the vision projector (`mmproj`); this test is **text-only**. No Hugging Face login is required if the checkpoint remains public. Revision SHA is recorded.

In [10]:
import sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub>=0.34,<1.0', 'hf_xet>=1.0', 'requests>=2.28',
], check=True)
from huggingface_hub import HfApi, hf_hub_download

REPO = 'huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF'
FILENAME = 'Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf'
MODEL_DIR = SCRATCH / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
revision = HfApi().model_info(REPO).sha
print('Hugging Face repo revision:', revision)
MODEL = Path(hf_hub_download(
    repo_id=REPO, filename=FILENAME, revision=revision,
    local_dir=str(MODEL_DIR),
))
size_gb = MODEL.stat().st_size / 1e9
print(f'Downloaded {MODEL.name}: {size_gb:.2f} GB')
if not (15.0 <= size_gb <= 18.5):
    raise RuntimeError('Unexpected GGUF size. Do not start the server; inspect this download.')
print('MODEL DOWNLOAD PASSED.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 8.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


Hugging Face repo revision: 3f101cd22b7999228bbd5d79a33975414eb9758b


Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_(…):   0%|          | 0.00/16.6G [00:00<?, ?B/s]

Downloaded Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf: 16.55 GB
MODEL DOWNLOAD PASSED.


## 4 — Start private inference (requested 32K context)

One slot; both T4s; GPU **layer split**, which places layers and their caches on separate GPUs. `f16` KV cache is the conservative compatibility baseline for our first test. Inference runs on `127.0.0.1:8080` **inside Kaggle only**. No public endpoint is created.

If the server logs a genuine CUDA OOM, **stop it**, change `CTX = 32768` to `CTX = 16384`, and rerun *this cell*. Do not mistake the *configured* context for verified end-to-end long-context capability; we will benchmark that after smoke testing.

In [11]:
import time
import requests

CTX = 32768  # Requested max context, NOT a claim that long-context quality is tested.
ALIAS = 'qwen3.8-27b-huihui-abliterated-q4'
PORT = 8080
BASE = f'http://127.0.0.1:{PORT}'
LOG = WORK / 'qwen38_abliterated_server.log'

if 'server_proc' in globals() and server_proc.poll() is None:
    raise RuntimeError('A server started by this notebook is already running; stop it first.')
try:
    already = requests.get(BASE + '/health', timeout=2)
except requests.RequestException:
    already = None
if already is not None:
    raise RuntimeError('Port 8080 already answers HTTP; stop the old server before continuing.')

cmd = [
    str(SERVER), '--model', str(MODEL), '--alias', ALIAS,
    '--host', '127.0.0.1', '--port', str(PORT),
    '--n-gpu-layers', '999', '--split-mode', 'layer', '--tensor-split', '1,1',
    '--ctx-size', str(CTX), '--parallel', '1', '--fit', 'off',
    '--batch-size', '512', '--ubatch-size', '128',
    '--cache-type-k', 'f16', '--cache-type-v', 'f16',
    '--flash-attn', 'auto', '--jinja', '--no-context-shift',
]
with LOG.open('w', encoding='utf-8') as log_file:
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print('Launching private server. Log:', LOG)
print('Configured context:', CTX)
print('llama.cpp commit:', commit)

ready = False
for attempt in range(96):  # Up to 8 minutes for model loading.
    if server_proc.poll() is not None:
        break
    try:
        r = requests.get(BASE + '/health', timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if attempt % 6 == 0:
        print(f'Waiting for model load... {(attempt + 1) * 5}s')
    time.sleep(5)
if not ready:
    print('Server exit code:', server_proc.poll())
    print('Server log tail:')
    subprocess.run(['tail', '-n', '110', str(LOG)], check=False)
    if server_proc.poll() is None:
        server_proc.terminate()
    raise RuntimeError('STOP: server did not become healthy; see log. OOM -> try 16K context.')

print('SERVER HEALTHY — local-only API:', BASE + '/v1')
subprocess.run([
    'nvidia-smi', '--query-gpu=index,name,memory.used,memory.total',
    '--format=csv,noheader',
], check=True)
print('Inspect log for CUDA0 and CUDA1 model-layer allocation before claiming dual-GPU use.')

Launching private server. Log: /kaggle/working/qwen38_abliterated_server.log
Configured context: 32768
llama.cpp commit: 2b129ccfa03aea330d2d9ac4650a10de393dbe3a
Waiting for model load... 5s
Waiting for model load... 35s
Waiting for model load... 65s
SERVER HEALTHY — local-only API: http://127.0.0.1:8080/v1
0, Tesla T4, 8117 MiB, 15360 MiB
1, Tesla T4, 9237 MiB, 15360 MiB
Inspect log for CUDA0 and CUDA1 model-layer allocation before claiming dual-GPU use.


## 5 — Actual model generation test (no generated code is executed)

This is a real `/v1/chat/completions` request, not merely a `/health` check. `reasoning_effort=none` keeps the first smoke test short. We print the response and returned token usage; inspect code manually.

In [12]:
import json
import time

started = time.monotonic()
response = requests.post(
    BASE + '/v1/chat/completions',
    json={
        'model': ALIAS,
        'messages': [
            {'role': 'system', 'content': 'You are a coding assistant. Return only Python code, no Markdown fences.'},
            {'role': 'user', 'content': 'Write a function add(a, b) that returns the sum of two numbers.'},
        ],
        'reasoning_effort': 'none',
        'temperature': 0,
        'max_tokens': 256,
        'stream': False,
    },
    timeout=900,
)
print('HTTP:', response.status_code)
response.raise_for_status()
data = response.json()
msg = data['choices'][0]['message']
text_output = msg.get('content') or ''
print('Latency (s):', round(time.monotonic() - started, 1))
print('Finish reason:', data['choices'][0].get('finish_reason'))
print('Usage:', json.dumps(data.get('usage'), ensure_ascii=False))
print('MODEL RESPONSE:\n', text_output[:6000])
generation_passed = ('def add(' in text_output and 'return ' in text_output)
print('GENERATION SMOKE:', 'PASS' if generation_passed else 'REVIEW RESPONSE — not a confirmed pass')
if not generation_passed:
    print('Reasoning content (if any):', str(msg.get('reasoning_content'))[:3000])
    raise RuntimeError('Inference responded but coding smoke test did not pass; share the response/log.')

HTTP: 200
Latency (s): 1.8
Finish reason: stop
Usage: {"completion_tokens": 13, "prompt_tokens": 48, "total_tokens": 61, "prompt_tokens_details": {"cached_tokens": 0}}
MODEL RESPONSE:
 def add(a, b):
    return a + b
GENERATION SMOKE: PASS


## 6 — Optional but important: one tool-call probe

We ask the model to call a **dummy function schema**, but DO NOT execute the tool or touch your filesystem. `tool_calls` is what a local VS Code agent needs for agentic work. If this fails, model inference can still be successful, but our later harness has a blocker to investigate.

In [13]:
tool_payload = {
    'model': ALIAS,
    'messages': [
        {'role': 'system', 'content': 'Use the supplied tool when it is available; do not invent lookup results.'},
        {'role': 'user', 'content': 'Call lookup_symbol for the exact symbol name parse_config. Do not answer from memory.'},
    ],
    'tools': [
        {
            'type': 'function',
            'function': {
                'name': 'lookup_symbol',
                'description': 'Look up a symbol in a local source index.',
                'parameters': {
                    'type': 'object',
                    'properties': {'name': {'type': 'string'}},
                    'required': ['name'],
                },
            },
        },
    ],
    'tool_choice': 'required',
    'reasoning_effort': 'none',
    'temperature': 0,
    'max_tokens': 512,
    'stream': False,
}
tool_call_passed = False
try:
    result = requests.post(BASE + '/v1/chat/completions', json=tool_payload, timeout=900)
    print('Tool probe HTTP:', result.status_code)
    result.raise_for_status()
    tool_msg = result.json()['choices'][0]['message']
    calls = tool_msg.get('tool_calls') or []
    print('Tool calls:', json.dumps(calls, indent=2, ensure_ascii=False)[:6000])
    if calls:
        fn = calls[0].get('function', {})
        try:
            args = json.loads(fn.get('arguments') or '{}')
        except json.JSONDecodeError:
            args = {}
        tool_call_passed = fn.get('name') == 'lookup_symbol' and args.get('name') == 'parse_config'
    print('TOOL-CALL PROBE:', 'PASS' if tool_call_passed else 'NOT PASSED — inspect before agent integration')
    if not tool_call_passed:
        print('Other response content:', str(tool_msg.get('content'))[:2500])
except requests.RequestException as exc:
    print('TOOL-CALL PROBE: NOT PASSED:', repr(exc))
print('No tools were executed; no local or remote project files were modified.')

Tool probe HTTP: 200
Tool calls: [
  {
    "type": "function",
    "function": {
      "name": "lookup_symbol",
      "arguments": "{\"name\":\"parse_config\"}"
    },
    "id": "VN4jvInjDe06P0ypipIat19ss9MujVIj"
  }
]
TOOL-CALL PROBE: PASS
No tools were executed; no local or remote project files were modified.


## 7 — Optional: test approximately 24K **actual input tokens**

The server is configured for 32K, but that is only a capacity setting. This optional probe constructs a synthetic source index, uses llama.cpp's tokenizer to target ~24K input tokens, and asks for a fact placed at the beginning. **It may take several minutes on two T4s.** Flip `RUN_LONG_CONTEXT_TEST` to `True` only when the basic tests pass and you want to spend the GPU time. This measures one recall task; it does not prove general 32K coding quality.

In [14]:
RUN_LONG_CONTEXT_TEST = False  # Change to True after the basic inference smoke test passes.
long_context_passed = None
long_context_prompt_tokens = None
if RUN_LONG_CONTEXT_TEST:
    target = 'amber otter 47'
    prefix = f'IMPORTANT TEST FACT: the deployment key is {target}.\n'
    suffix = '\nWhat was the deployment key at the very beginning? Answer with the key only.'
    repeats = 1800
    for _ in range(5):
        filler = '\n'.join(
            f'synthetic/file_{i:05d}.py: def helper_{i}(): return {i % 97}'
            for i in range(repeats)
        )
        prompt = prefix + filler + suffix
        tok = requests.post(
            BASE + '/tokenize', json={'content': prompt}, timeout=120,
        )
        tok.raise_for_status()
        actual = len(tok.json()['tokens'])
        print('Synthetic prompt tokens:', actual, 'at repeats:', repeats)
        if 21000 <= actual <= 27000:
            break
        repeats = max(100, int(repeats * 24000 / max(actual, 1)))
    if not 21000 <= actual <= 27000:
        raise RuntimeError('Could not construct a 21K–27K token prompt; do not send an over-context request.')
    t0 = time.monotonic()
    test = requests.post(
        BASE + '/v1/chat/completions',
        json={
            'model': ALIAS,
            'messages': [{'role': 'user', 'content': prompt}],
            'reasoning_effort': 'none', 'temperature': 0,
            'max_tokens': 128, 'stream': False,
        },
        timeout=1800,
    )
    test.raise_for_status()
    context_data = test.json()
    context_answer = context_data['choices'][0]['message'].get('content') or ''
    long_context_prompt_tokens = context_data.get('usage', {}).get('prompt_tokens')
    long_context_passed = target in context_answer.lower()
    print('Context test latency (seconds):', round(time.monotonic() - t0, 1))
    print('Actual API-reported prompt tokens:', long_context_prompt_tokens)
    print('Answer:', context_answer[:1000])
    print('LONG-CONTEXT RECALL:', 'PASS' if long_context_passed else 'NOT PASSED')
else:
    print('Long-context test SKIPPED. Configured 32K context is not yet performance-verified.')

Long-context test SKIPPED. Configured 32K context is not yet performance-verified.


## 8 — Capture lightweight diagnostics for our next step

Saved under `/kaggle/working/` so it can be copied without sharing your model file. The log may contain prompts or endpoint details; **review/redact it before sharing**. A successful `/health` check alone does **not** prove code quality, tool calling, or a working VS Code tunnel.

In [15]:
import json
import datetime

gpu_usage = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=True,
).stdout.strip()
report = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'repo': REPO,
    'revision': revision,
    'file': FILENAME,
    'model_size_gb': round(size_gb, 2),
    'llama_cpp_commit': commit,
    'requested_ctx_tokens': CTX,
    'engine': str(SERVER),
    'server_healthy': bool(server_proc.poll() is None and requests.get(BASE + '/health', timeout=5).ok),
    'generation_smoke_passed': bool(globals().get('generation_passed', False)),
    'tool_call_probe_passed': bool(globals().get('tool_call_passed', False)),
    'long_context_recall_passed': globals().get('long_context_passed', None),
    'long_context_api_prompt_tokens': globals().get('long_context_prompt_tokens', None),
    'gpu_usage': gpu_usage,
    'server_log': str(LOG),
    'note': 'Real runtime results on this Kaggle session only; no VS Code connection established.',
}
REPORT = WORK / 'qwen38_abliterated_runtime_report.json'
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('\nLAST SERVER LOG LINES (review/redact before sharing):')
subprocess.run(['tail', '-n', '45', str(LOG)], check=False)
print('\nFiles to share after redaction:', REPORT, LOG)

{
  "timestamp_utc": "2026-09-27T06:14:04.446343+00:00",
  "repo": "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF",
  "revision": "3f101cd22b7999228bbd5d79a33975414eb9758b",
  "file": "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf",
  "model_size_gb": 16.55,
  "llama_cpp_commit": "2b129ccfa03aea330d2d9ac4650a10de393dbe3a",
  "requested_ctx_tokens": 32768,
  "engine": "/kaggle/tmp/qwen38-abliterated/llama.cpp/build/bin/llama-server",
  "server_healthy": true,
  "generation_smoke_passed": true,
  "tool_call_probe_passed": true,
  "long_context_recall_passed": null,
  "long_context_api_prompt_tokens": null,
  "gpu_usage": "0, Tesla T4, 8129 MiB, 15360 MiB\n1, Tesla T4, 9247 MiB, 15360 MiB",
  "server_log": "/kaggle/working/qwen38_abliterated_server.log",
  "note": "Real runtime results on this Kaggle session only; no VS Code connection established."
}

LAST SERVER LOG LINES (review/redact before sharing):
0.00.000.754 I srv  llama_server: initializing ...
0.00.286.543 I cmn  common_par

## 9 — Stop the inference server when finished

Stops only the `llama-server` process started by this notebook. Kaggle itself will continue until you **Stop Session** in the web UI. The Kaggle GPU runtime, model weights in `/kaggle/tmp/`, and session quota are separate from this process.

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=10)
    print('Model server stopped. You can now Stop Session in Kaggle.')
else:
    print('No notebook-managed model server is running.')